# **MCP - Model Context Protocol based Agent Applicatiokn**

In [ ]:
!pip -q install chromadb openai "mcp>=1.28, <2"
# install mcp version greater than or euqal to 1.28 and less than 2

In [ ]:
# create basic configurations
from google.colab import userdata, drive
import os

# get the api key from secrets
api_key = userdata.get('my_key')

# mount the google drive and get the path to the file
drive.mount("/content/drive")
HR_FILE = '/content/drive/MyDrive/Projects/companyhrdoc.txt'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile mcp_hr_server.py

# create the mcp server
# this server exposes two tools - retrieve_hr_policy, calculator
import re
import ast
import operator
import chromadb
from mcp.server.fastmcp import FastMCP

# step 1 : Create the mcp server, define the port and the host ip address
mcp = FastMCP("HR Policy Tools", host = "127.0.0.1", port = 8000)

# Step 2: create a new chromadb database
DB_PATH = '/content/mcp_hr_chroma_db'
COLLECTION_NAME = 'mcp_hr_policy'

# Define the HR file path inside the script file
HR_FILE = '/content/drive/MyDrive/Projects/companyhrdoc.txt'

# create a chromadb persistent client (will save the database
# in an sqlite file)
chroma_client = chromadb.PersistentClient(path = DB_PATH)

# create a collection
collection = chroma_client.get_or_create_collection(
    name = COLLECTION_NAME,
    metadata = {'description' : 'HR policy documets for MCP'},
)

# Step 3: Load the created chunks from the hr policy document
def load_and_index_policy():
  '''
  Load the HR file, split it into sections and index it
  '''
  # open the file and save the contents inside a string variable
  with open(HR_FILE, 'r', encoding = 'utf-8') as file:
    text = file.read()

  # split the document at "Section 1:" pattern
  chunks = re.split(r'(?=Section \d+:)', text)

  # clean the chunks by stripping the white space
  chunks = [chunk.strip() for chunk in chunks if chunk.strip()]

  # generate unique ids for each chunk
  ids = [f"hr_section_{i}" for i in range(len(chunks))]

  # add the chunks and its ids into collection
  collection.upsert(ids= ids, documents = chunks)

  return len(chunks)

# Step 4: index the policy chunks when the server starts
indexed_chunks = load_and_index_policy()

# Step 5: declare the first mcp tool - search the hr policy from chromadb
@mcp.tool()
def retrieve_hr_policy(query : str) -> str:
  '''
  search the company's HR Policy for information relevant to a question
  Use this tool for questions about leave, eligibility, entitlements,
  rollover limits and other details contained in the HR policy
  The returned text contains relevant document sections,
  but not a guaranteed complete answer
  '''

  results = collection.query(query_texts = [query], n_results = 3)
  documents = results.get("documents",[[]])[0]
  if not documents:
    return "No relevant HR Policy sections were found."
  # if valid documents
  return "\n\n ---Retrieved Policy Sections--- \n\n".join(documents)

# Step 6: declare the second mcp tool - calculator

ALLOWED_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}

def safe_eval(node):
  if isinstance(node, ast.Expression):
    return safe_eval(node.body)
  # only plain numbers are allowed
  if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
     and not isinstance(node.value, bool):
    return node.value
  if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_OPS:
    left = safe_eval(node.left)
    right = safe_eval(node.right)
    if isinstance(node.op, ast.Pow) and abs(right) > 10:
      raise ValueError("exponent too large")
    return ALLOWED_OPS[type(node.op)](left, right)
  if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_OPS:
    return ALLOWED_OPS[type(node.op)](safe_eval(node.operand))
  raise ValueError("unsupported expression")

@mcp.tool()
def calculator(expression: str) -> str:
  '''
  Calculate a basic arithmetic expression such as 4 * 0.25.
  Supports + - * / and ** with numbers only.
  Use this tool for numerical calculations.
  '''
  if len(expression) > 100:
    return "calculation error : expression too long"
  try:
    tree = ast.parse(expression, mode="eval")
    return str(safe_eval(tree))
  except Exception as error:
    return f"calculation error : {error}"

if __name__ == '__main__':
  mcp.run(transport='streamable-http')

Overwriting mcp_hr_server.py


In [ ]:
# start the server and run in a seperate colab cell
import subprocess
import sys
import time
import socket

try:
  mcp_process.terminate()
  time.sleep(2)
except NameError:
  pass

# write server logs
server_log = open("mcp_server.log", "w")

# create a subprocess for our mcp server
mcp_process = subprocess.Popen([sys.executable,
                                'mcp_hr_server.py'],
                               stdout=server_log,
                               stderr=subprocess.STDOUT)

# try to start the server - 30 attempts maximum
for attempt in range(30):
  try:
    with socket.create_connection(("127.0.0.1",8000), timeout=1):
      print("MCP Server successfully launched and running")
      break
  except OSError:
    time.sleep(1)
else:
    print("MCP Server failed to start")
    print(open("mcp_server.log").read())

MCP Server successfully launched and running


In [ ]:
# Test our mcp server by querying for the list of tools in the server
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# define an asynchronous fn to connect to the mcp server
async def test_mcp_connection():
  # connect to the mcp server over http
  async with streamable_http_client(
      "http://127.0.0.1:8000/mcp"
  ) as (read, write, _):
   async with ClientSession(read, write) as session:
    await session.initialize()

    # once conncected, discover the tools served by the mcp server
    tools = await session.list_tools()
    print("Available tools in this server:")
    for tool in tools.tools:
      print(tool.name)

# run the fucntion
await test_mcp_connection()

Available tools in this server:
retrieve_hr_policy
calculator


In [ ]:
# connect to our running mcp server, discover the tools
# convert the tools list into a list that llm can understand
# then let the llm decide which tool to call and when to call
# and finally
import json
from openai import AsyncOpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

client = AsyncOpenAI(api_key = api_key)

INSTRUCTIONS = (
    "You are an HR policy assistant. "
    "Use retrieve_hr_policy for company policy questions. "
    "Use calculator for arithmetic calculations. "
    "Answer using retrieved policy evidence. "
    "If information is missing, say so politely. "
    "Do not make up information."
)

async def ask_hr_agent(question):
  # connect to the running mcp server over http with read, write stream
  async with streamable_http_client("http://127.0.0.1:8000/mcp") as (read, write, _):

   async with ClientSession(read, write) as session:

    await session.initialize()

    # once conncected, discover the tools on the mcp server
    mcp_tools = await session.list_tools()

    # convert the mcp tool definitions into LLm function definitions
    # so that it can understand
    llm_tools =[{
        'type':'function',
        'name': tool.name,
        'description':tool.description or "",
        'parameters':tool.inputSchema
    } for tool in mcp_tools.tools]

    # start the conversation with user's query
    conversation = [{'role':'user', 'content':question}]

    # define the agent loop: ask the llm to execute the requested tools
    # then send the result/evidence back to the llm itself to get the final answer
    for step in range(10) :
      response = await client.responses.create(
          model = 'gpt-6-luna',
          instructions = INSTRUCTIONS,
          input = conversation,
          tools = llm_tools
      )
      # preserve the models response and tool requests(context memory)
      conversation.extend(
          item.model_dump(exclude_none=True)
          for item in response.output
      )

      # find any function calls in the response requested by the model
      tool_calls = [
          item for item in response.output
          if item.type == 'function_call'
      ]

      # if no tools are requested, this loop is the final answer loop
      if not tool_calls:
        print(response.output_text)
        return response.output_text

      # if tool_calls are present, then execute each requested tool through mcp
      for call in tool_calls:
        arguments = json.loads(call.arguments)
        tool_result = await session.call_tool(call.name, arguments)

        # convert the mcp tool output into readable text
        if tool_result.isError:
          result_text = "Tool Execution failed"
        else:
          result_text = "\n".join(item.text
                                  for item in tool_result.content
                                  if hasattr(item, 'text'))
        # return the tool result back to the llm
        conversation.append({
              'type' : 'function_call_output',
              'call_id' : call.call_id,
              'output' : result_text
        })

  raise RuntimeError  ('Maximum number of agent steps reached')

In [ ]:
result = await ask_hr_agent(
    "According to the HR policy, what is the annual PTO entitlement."
    'for employees with 4 years of service? calculate the total PTO'
    'for 3 such employees.'
)

Employees with 4 years of service accrue **20 days of PTO per year**. For 3 such employees, the combined annual entitlement is **60 days of PTO**.
